In [ ]:
import warnings
import pandas as pd
import gtfs_kit as gk
from typing import List, Tuple, Set
import copy
from pathlib import Path
import re


In [ ]:
gtfs_output_path = Path("/home/simpal/otp/data")

gtfs_year = 2025
gtfs_root = Path("/home/simpal/O/sharing-trans-data/GTFS Data/CLEAN - GTFS DATA/" + str(gtfs_year))

if not gtfs_root.is_dir():
    print("INPUT ERROR: Directory not found: " + str(gtfs_root))
# Recursively find zip files
gtfs_files = list(gtfs_root.rglob("*.zip"))

# Extract YYYYMMDD from filename
def extract_date(path):
    match = re.search(r"\d{8}", path.name)
    return pd.to_datetime(match.group(), format="%Y%m%d") if match else None


gtfs_release = (
    pd.DataFrame({
        "path": gtfs_files,
        "file": [p.name for p in gtfs_files],
        "date": [extract_date(p) for p in gtfs_files],
    })
    .dropna(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
    .assign(
        date_end = lambda df: df["date"].shift(-1),
        date_days = lambda df: (df["date"] - pd.Timestamp("1970-01-01")).dt.days,
        date_end_days = lambda df: (df["date_end"] - pd.Timestamp("1970-01-01")).dt.days,
    )
)


print(f"Total number GTFS files: {len(gtfs_release)}")

In [ ]:
gtfs_list = {}
#Read all gtfs_files
for _, row in gtfs_release.iterrows():
    print(f"Reading GTFS file: {row['file']}")

    feed = gk.feed.read_feed(row["path"], dist_units = "m") #Not sure what dist_unit is.

    gtfs_list[row["file"]] = feed


In [ ]:
#Function to trunceate GTFS feed at specified date
#Cutoff  date will not be included
def truncate_feed_to_date(feed_i, cutoff_date):
    cutoff_str = cutoff_date.strftime("%Y%m%d")

    if feed_i.calendar is not None:
        cal = feed_i.calendar.copy()
        cal = cal[cal.start_date < cutoff_str]
        cal.loc[cal.end_date >= cutoff_str, "end_date"] = cutoff_str
        feed_i.calendar = cal
        del cal

    if feed_i.calendar_dates is not None:
        cd = feed_i.calendar_dates.copy()
        cd = cd[cd.date < cutoff_str]
        feed_i.calendar_dates = cd
        del cd

    #filter trips using valid service_id
    valid_service_ids = set()
    if feed_i.calendar is not None:
        valid_service_ids.update(feed_i.calendar.service_id.unique())

    if feed_i.calendar_dates is not None:
        valid_service_ids.update(feed_i.calendar_dates.service_id.unique())

    feed_i.trips = feed_i.trips[
        feed_i.trips.service_id.isin(valid_service_ids)
    ]

    #restrict_to_trips: Build a new feed by restricting this one to only the stops, trips, shapes, etc. used by the trips of the given IDs. Return the resulting feed.
    feed_trunc = gk.miscellany.restrict_to_trips(feed_i, feed_i.trips.trip_id.tolist())


    return feed_trunc

In [ ]:
#truncating all files
for i, row in gtfs_release.iterrows():
    file_key = row["file"]
    cutoff = row["date_end"]

    if pd.isna(cutoff):
        continue

    print(f"Truncating {file_key} to {cutoff.date()}")
    gtfs_list[file_key] = truncate_feed_to_date(gtfs_list[file_key], cutoff)


In [ ]:
# Configuration: primary table name as key, with id column stored in id_col
ID_CONFIG = {
    "stops": {
        "id_col": "stop_id",
        "identity_cols": ["stop_lat", "stop_lon", "stop_name", "stop_id"], #stops only table with is primary key included in identity_cols. This is because stops are often moved somewhat, but I've been promised they don't change stop_id unless they move it more than 40 meters.
        "foreign_keys": [
            ("stop_times", "stop_id"),
            ("transfers", "from_stop_id"),
            ("transfers", "to_stop_id"),
        ],
    },

    "shapes": {
        "id_col": "shape_id",
        "identity_cols": ["shape_pt_lat", "shape_pt_lon", "shape_pt_sequence"],
        "foreign_keys": [
            ("trips", "shape_id"),
        ],
    },

    "agency": {
        "id_col": "agency_id",
        "identity_cols": ["agency_name", "agency_timezone"],
        "foreign_keys": [
            ("routes", "agency_id"),
        ],
    },

    "routes": {
        "id_col": "route_id",
        "identity_cols": ["agency_id", "route_short_name", "route_type"],
        "foreign_keys": [
            ("trips", "route_id"),
            ("transfers", "from_route_id"),
            ("transfers", "to_route_id"),
        ],
    },

    "trips": {
        "id_col": "trip_id",
        "identity_cols": ["service_id", "trip_headsign", "trip_short_name", "direction_id"],
        "foreign_keys": [
            ("stop_times", "trip_id"),
            ("transfers", "from_trip_id"),
            ("transfers", "to_trip_id"),
        ],
    },
}

ID_CONFIG_service_id = {
    "calendar": {
        "id_col": "service_id",
        "identity_cols": [
            "monday", "tuesday", "wednesday", "thursday", "friday",
            "saturday", "sunday", "start_date", "end_date"
        ],
        "foreign_keys": [
            ("trips", "service_id"),
            ("calendar_dates", "service_id"),
        ],
    }
}

In [ ]:
#Merge all seperate feeds into one
print("\nMerging all feeds into combined GTFS feed...")
if "combined_feed" in globals(): #in case of rerun
    del combined_feed

feed_names = list(gtfs_list.keys())
combined_feed = copy.deepcopy(gtfs_list[feed_names[0]])
print(f"Starting with base feed: {feed_names[0]}")

# Add feed_id to the inital feed tables (same way you do for subsequent feeds)
for table in [
    "agency", "routes", "stops", "trips", "stop_times",
    "calendar", "calendar_dates", "shapes", "transfers"
]:
    df = getattr(combined_feed, table, None)
    setattr(combined_feed, table, df.assign(feed_id=feed_names[0]))

for feed_name in feed_names[1:]:
    print(f"Merging feed: {feed_name}")
    feed_to_merge = gtfs_list[feed_name]

    combined_feed.agency = pd.concat([combined_feed.agency, feed_to_merge.agency.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.routes = pd.concat([combined_feed.routes, feed_to_merge.routes.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.stops = pd.concat([combined_feed.stops, feed_to_merge.stops.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.trips = pd.concat([combined_feed.trips, feed_to_merge.trips.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.stop_times = pd.concat([combined_feed.stop_times, feed_to_merge.stop_times.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.calendar = pd.concat([combined_feed.calendar, feed_to_merge.calendar.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.calendar_dates = pd.concat([combined_feed.calendar_dates, feed_to_merge.calendar_dates.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.shapes = pd.concat([combined_feed.shapes, feed_to_merge.shapes.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.transfers = pd.concat([combined_feed.transfers, feed_to_merge.transfers.assign(feed_id = feed_name)], ignore_index=True)

    del gtfs_list[feed_name], feed_to_merge


print(f"\nMerge complete! Combined feed statistics:")
if combined_feed.agency is not None:
    print(f"  Agencies: {len(combined_feed.agency)}")
if combined_feed.routes is not None:
    print(f"  Routes: {len(combined_feed.routes)}")
if combined_feed.stops is not None:
    print(f"  Stops: {len(combined_feed.stops)}")
if combined_feed.trips is not None:
    print(f"  Trips: {len(combined_feed.trips)}")
if combined_feed.stop_times is not None:
    print(f"  Stop times: {len(combined_feed.stop_times)}")
if combined_feed.calendar is not None:
    print(f"  Calendar entries: {len(combined_feed.calendar)}")
if combined_feed.shapes is not None:
    print(f"  Shape points: {len(combined_feed.shapes)}")

In [ ]:
len(gtfs_list[feed_names[0]].calendar)

In [ ]:
combined_feed_org = copy.deepcopy(combined_feed)

In [ ]:
def find_conflicting_ids(feed, id_col, primary_table, identity_cols):
    df = getattr(feed, primary_table, None)
    if df is None:
        raise ValueError(f"Error: {primary_table} not found in feed")

    if any(col not in df.columns for col in identity_cols):
        raise ValueError(f"Error: one or more identity_cols not found in {primary_table}")

    use_identity_cols = identity_cols + [id_col]

    # Group by ID and count unique definitions
    id_definitions = (
        df
        .drop_duplicates(subset=use_identity_cols)
        .groupby(id_col)
        .size()
    )
    conflicting = set(id_definitions[id_definitions > 1].index)
    return conflicting


def apply_prefix_to_feed(feed, id_col, conflicting_ids, foreign_keys, primary_table):

    if not conflicting_ids:
        return

    # All tables/columns to update (primary + foreign keys)
    tables_to_update = [(primary_table, id_col)]
    tables_to_update.extend(foreign_keys)

    # Collect all conflicting IDs that exist in this feed
    feed_conflicting_ids = set()

    for table_name, col_name in tables_to_update:
        df = getattr(feed, table_name, None)
        if df is None or col_name not in df.columns:
            continue

        existing_ids = set(df[col_name].dropna().unique())
        feed_conflicting_ids.update(existing_ids & conflicting_ids)


    # Apply prefix to all tables
    for table_name, col_name in tables_to_update:
        df = getattr(feed, table_name, None)
        if df is None or col_name not in df.columns:
            continue

        mask = df[col_name].isin(feed_conflicting_ids)
        #add prefix from feed_id (feed_id was added to the combined_feed above)
        if mask.any():
            if "feed_id" not in df.columns:
                raise ValueError(f"{table_name} is missing feed_id")

            df.loc[mask, col_name] = (
                df.loc[mask, "feed_id"].astype(str)
                .str.replace("GTFS_", "", regex=False)
                .str.replace(".zip", "", regex=False)
                + "_"
                + df.loc[mask, col_name].astype(str)
            )

In [ ]:
#add prefix to non-unique service_id. Rest ID are prefixed later, but since service_id is especially inconsistent across feed over time, prefix is added before trying to drop duplicates later.
primary_table, config = next(iter(ID_CONFIG_service_id.items()))
print(f"\nProcessing {primary_table}...")
conflicting = find_conflicting_ids(
    combined_feed,
    config["id_col"],
    primary_table,
    config["identity_cols"]
)
print(f"  Found {len(conflicting)} conflicting {config["id_col"]} values")
if conflicting:
    apply_prefix_to_feed(combined_feed, config["id_col"], conflicting, config["foreign_keys"], primary_table)

    print(f"    Prefixed conflicting {config["id_col"]} in all feeds")

In [ ]:
combined_feed = copy.deepcopy(combined_feed_org)

In [ ]:
def deduplicate_feed(feed: gk.feed.Feed, id_col: str, primary_table: str, identity_cols: List[str],
                     foreign_keys: List[Tuple[str, str]]) -> int:
    df_primary = getattr(feed, primary_table, None)
    if df_primary is None or df_primary.empty:
        raise ValueError(f"Error: {primary_table} not found in feed")

    initial_count = len(df_primary)

    # Filter identity_cols to those present in the dataframe
    use_identity_cols = [c for c in identity_cols if c in df_primary.columns and c != id_col]

    if not use_identity_cols:
        raise ValueError(f"No identity columns found for {primary_table}")

    if primary_table in ["shapes", "stop_times"]:
        if primary_table == "shapes":
            sequence_cols = "shape_pt_sequence"
        if primary_table == "stop_times":
            sequence_cols = "stop_sequence"
            
        df_primary[id_col] = (
            df_primary['feed_id'].astype(str)
            .str.replace("GTFS_", "", regex=False)
            .str.replace(".zip", "", regex=False)
            + "_"
            + df_primary[id_col]
        )
        # For sequence tables: create signature from all rows grouped by ID
        df_primary = df_primary.sort_values([id_col, sequence_cols])

        # Create row-level signature by concatenating identity columns
        df_primary['_row_sig'] = pd.util.hash_pandas_object(
            df_primary[use_identity_cols],
            index = False
        )

        # Group and concatenate row signatures into single signature per ID
        signatures = (
            df_primary
            .groupby(id_col)["_row_sig"]
            .apply(tuple)
            .map(hash)
            .rename("_signature")
            .reset_index()
        )
        df_primary.drop(columns=["_row_sig"], inplace=True)
        # Map signature to canonical (minimum) ID
        canonical_map = (
            signatures
            .groupby('_signature')[id_col]
            .min()
        )

        # Create ID to canonical ID mapping. For foreign key
        id_to_canonical = (
            signatures
            .set_index(id_col)['_signature']
            .map(canonical_map)
        )

        # Get set of canonical IDs
        canonical_ids = canonical_map.values

        # Update primary table: keep only rows with canonical IDs
        df_primary = (
            df_primary[df_primary[id_col].isin(canonical_ids)]
            .reset_index(drop=True)
            .drop_duplicates(subset=use_identity_cols + [id_col], keep="first")
        )
        setattr(feed, primary_table, df_primary)

    elif primary_table == "trips":
        #df_primary = df_primary.sort_values(["trip_id"])
        df_st = feed.stop_times.sort_values(["trip_id", "stop_sequence"])

        df_st['_row_sig'] = pd.util.hash_pandas_object(
            df_st[["stop_id", "arrival_time", "departure_time"]],
            index = False
        )

        pattern = (
            df_st.groupby("trip_id")["_row_sig"]
            .apply(tuple)
            .map(hash)
            .rename("pattern_sig")
        )
        df_st = df_st.drop(columns=["_row_sig"]) # save memory
        df_primary["pattern_sig"] = df_primary["trip_id"].map(pattern)

        df_primary["trip_sig"] = pd.util.hash_pandas_object(
            df_primary[["route_id", "service_id", "direction_id", "shape_id", "pattern_sig"]],
            index = False
        )

        canonical = (
            df_primary
            .groupby("trip_sig")["trip_id"]
            .min()
        )

        trip_map = (
            df_primary.set_index("trip_id")["trip_sig"]
            .map(canonical)
        )

        df_primary["trip_id"] = df_primary["trip_id"].map(trip_map).fillna(df_primary["trip_id"])
        df_st["trip_id"] = df_st["trip_id"].map(trip_map).fillna(df_st["trip_id"])

        df_primary = df_primary.drop_duplicates("trip_id")

        df_st = (
            df_st
            .sort_values(["trip_id", "stop_sequence"])
            .drop_duplicates(["trip_id", "stop_sequence"])
        )
        df_primary = (
            df_primary
            .sort_values(["trip_id"])
            .drop_duplicates(["trip_id"])
            .drop(columns=["pattern_sig", "trip_sig"], errors="ignore")
        )

        df_primary = df_primary.drop(columns=["pattern_sig", "trip_sig"], errors="ignore")

        feed.trips = df_primary.reset_index(drop=True)
        feed.stop_times = df_st.reset_index(drop=True)

        final_count = len(df_primary)
        duplicates_removed = initial_count - final_count

        return duplicates_removed #don't run foreign key loop has it has been done manually for trips

    elif primary_table == "stops": #coordinate of stops sometimes changes a little bit. I've been told they keep stop_id consistent (!) and only change it when it is moved more than 40 m. The "within 40m=stop_id" is not consistent.
        #df_primary = df_primary.sort_values(id_col)

        lat_threshold = 0.0003592535
        lon_threshold = 0.00064109755

        max_lat_delta = (
            df_primary.groupby("stop_id")["stop_lat"]
            .transform(lambda x: (x.mean() - x).abs().max()) #stop distance from average coordinate
        )
        max_lon_delta = (
            df_primary.groupby("stop_id")["stop_lon"]
            .transform(lambda x: (x.mean() - x).abs().max())
        )
        df_primary["stable_loc"] = (max_lat_delta < lat_threshold) &  (max_lon_delta < lon_threshold)  #~40m (~56.6m in diagonal movement) at 56N

        stable_means = (
            df_primary.loc[df_primary["stable_loc"]]
            .groupby("stop_id", as_index=True)[["stop_lat", "stop_lon"]]
            .mean()
        )

        # write back means only for stable rows
        stable_mask = df_primary["stable_loc"]
        df_primary.loc[stable_mask, "stop_lat"] = df_primary.loc[stable_mask, "stop_id"].map(stable_means["stop_lat"])
        df_primary.loc[stable_mask, "stop_lon"] = df_primary.loc[stable_mask, "stop_id"].map(stable_means["stop_lon"])

        # warning flags
        if (~df_primary['stable_loc']).any():
            print(
                "Warning: stop_id with max delta lat/lon above 40m threshold:\n",
                df_primary.loc[df_primary['stable_loc'], ["stop_id"]].drop_duplicates(),
                "\nDuplicated stop_id with lat/lon differences > 40, will get new stop_id."
            )

        #drop duplicates with same stop_id/_lat/_lon. lat/lon has been average by stop_id if within 40m. Duplicated stop_id with delta lat/lon, will get new stop_id below.
        df_primary = df_primary.drop_duplicates(subset=["stop_id", "stop_lat", "stop_lon"], inplace=False)


        feed.stops = df_primary.reset_index(drop=True)
        final_count = len(df_primary)
        duplicates_removed = initial_count - final_count
        return duplicates_removed


    else:
        # For simple tables: group by identity columns directly
        #df_primary = df_primary.sort_values(id_col)

        # Map each unique combination of identity cols to canonical (minimum) ID
        canonical_df = (
            df_primary
            .groupby(use_identity_cols, dropna=False)[id_col]
            .min()
            .reset_index()
        )

        # Create mapping from all IDs to canonical IDs
        id_to_canonical = (
            df_primary[[id_col] + use_identity_cols]
            .merge(canonical_df, on=use_identity_cols, suffixes=('', '_canonical'))
            .drop_duplicates(subset=[id_col, f'{id_col}_canonical'])
            .set_index(id_col)[f'{id_col}_canonical']
        ) #For foreign key #For foreign key

        # Update primary table: keep only canonical rows
        canonical_ids = canonical_df[id_col]

        df_primary = (
            df_primary[df_primary[id_col].isin(canonical_ids)]
            .reset_index(drop=True)
            .drop_duplicates(subset=use_identity_cols + [id_col], keep="first")
        )
        setattr(feed, primary_table, df_primary)

    # Update all foreign key references
    for fk_table, fk_col in foreign_keys:
        fk_df = getattr(feed, fk_table, None)
        if fk_df is None or fk_col not in fk_df.columns:
            warnings.warn(f"Warning: Foreign key column {fk_col} not found in {fk_table}. Skipping foreign key update.")
            continue

        # Map foreign keys to canonical IDs
        fk_df[fk_col] = fk_df[fk_col].map(id_to_canonical).fillna(fk_df[fk_col])
        setattr(feed, fk_table, fk_df)

    final_count = len(getattr(feed, primary_table))
    duplicates_removed = initial_count - final_count

    return duplicates_removed



In [ ]:
tables_to_deduplicate = ['stops', 'shapes', 'routes', 'agency', 'trips', 'stop_times']

print("Starting deduplication process...")
print(f"\nBefore deduplication:")
for table in tables_to_deduplicate:
    if getattr(combined_feed, table, None) is not None:
        print(f"  {table}: {len(getattr(combined_feed, table, None))}")

# Apply deduplication for each ID type
for primary_table, config in ID_CONFIG.items(): #Important calendar and calendar_dates are not included in removing duplicates!
    print(f"\nDeduplicating {primary_table}...")

    removed = deduplicate_feed(
        combined_feed,
        config["id_col"],
        primary_table,
        config["identity_cols"],
        config["foreign_keys"]
    )

    df = getattr(combined_feed, primary_table, None)
    if df is not None:
        print(f"  {primary_table}: removed {removed} duplicates, {len(df)} remaining")

print("\n" + "=" * 50)
print("Deduplication complete!")
print(f"\nAfter deduplication:")
for table in tables_to_deduplicate:
    df = getattr(combined_feed, table, None)
    if df is not None:
        print(f"  {table}: {len(df)}")

del df

In [ ]:
# Apply to prefix to all ID types
for primary_table, config in ID_CONFIG.items():
    if config['primary_table'].isin(['calendar', 'calendar_dates', 'shapes', 'stop_times']):
        continue
    print(f"\nProcessing {primary_table}...")
    conflicting = find_conflicting_ids(
        combined_feed,
        config["id_col"],
        primary_table,
        config["identity_cols"]
    )
    print(f"  Found {len(conflicting)} conflicting {config["id_col"]} values")
    if conflicting:
        apply_prefix_to_feed(combined_feed, config["id_col"], conflicting, config["foreign_keys"], config["primary_table"])
        print(f"    Prefixed conflicting {config["id_col"]} in all feeds")

In [ ]:
df_primary = combined_feed.trips.copy()
id_col = "trip_id"
primary_table = "trips"
identity_cols = ["service_id", "trip_headsign", "trip_short_name", "direction_id"]
foreign_keys = [("stop_times", "trip_id"), ]
df_primary


In [ ]:
#df_primary = df_primary.sort_values(["trip_id"])
df_st = feed.stop_times.sort_values(["trip_id", "stop_sequence"])

df_st['_row_sig'] = pd.util.hash_pandas_object(
    df_st[["stop_id", "arrival_time", "departure_time"]],
    index = False
)

pattern = (
    df_st.groupby("trip_id")["_row_sig"]
    .apply(tuple)
    .map(hash)
    .rename("pattern_sig")
)
df_st = df_st.drop(columns=["_row_sig"]) # save memory
df_primary["pattern_sig"] = df_primary["trip_id"].map(pattern)

df_primary["trip_sig"] = pd.util.hash_pandas_object(
    df_primary[["route_id", "service_id", "direction_id", "shape_id", "pattern_sig"]],
    index = False
)

canonical = (
    df_primary
    .groupby("trip_sig")["trip_id"]
    .min()
)

trip_map = (
    df_primary.set_index("trip_id")["trip_sig"]
    .map(canonical)
)

df_primary["trip_id"] = df_primary["trip_id"].map(trip_map).fillna(df_primary["trip_id"])
df_st["trip_id"] = df_st["trip_id"].map(trip_map).fillna(df_st["trip_id"])

df_primary = df_primary.drop_duplicates("trip_id")

df_st = (
    df_st
    .sort_values(["trip_id", "stop_sequence"])
    .drop_duplicates(["trip_id", "stop_sequence"])
)
df_primary = (
    df_primary
    .sort_values(["trip_id"])
    .drop_duplicates(["trip_id"])
    .drop(columns=["pattern_sig", "trip_sig"], errors="ignore")
)

df_primary = df_primary.drop(columns=["pattern_sig", "trip_sig"], errors="ignore")

feed.trips = df_primary.reset_index(drop=True)
feed.stop_times = df_st.reset_index(drop=True)

final_count = len(df_primary)
duplicates_removed = initial_count - final_count

In [ ]:
df_primary = combined_feed.shapes.copy()
id_col = "shape_id"
primary_table = "shapes"
identity_cols = ["shape_pt_lat", "shape_pt_lon", "shape_pt_sequence"]
foreign_keys = [("trips", "shape_id"),]
df_primary

In [ ]:

use_identity_cols = identity_cols

In [ ]:
df_primary.info()

In [ ]:
sequence_cols = ["shape_pt_sequence"]

# For sequence tables: create signature from all rows grouped by ID
df_sorted = df_primary.sort_values([id_col] + sequence_cols)

In [ ]:
df_sorted[id_col] = (
    df_sorted['feed_id'].astype(str)
    .str.replace("GTFS_", "", regex=False)
    .str.replace(".zip", "", regex=False)
    + "_"
    + df_sorted[id_col]
)

In [ ]:
# Create row-level signature by concatenating identity columns
df_sorted['_row_sig'] = pd.util.hash_pandas_object(
    df_sorted[use_identity_cols],
    index = False
)

In [ ]:
df_sorted

In [ ]:
# Group and concatenate row signatures into single signature per ID
signatures = (
    df_sorted
    .groupby(id_col)["_row_sig"]
    .apply(tuple)
    .map(hash)
    .rename("_signature")
    .reset_index()
)
signatures

In [ ]:
# Map signature to canonical (minimum) ID
canonical_map = (
    signatures
    .groupby('_signature')[id_col]
    .min()
)
canonical_map

In [ ]:
# Create ID to canonical ID mapping
id_to_canonical = (
    signatures
    .set_index(id_col)['_signature']
    .map(canonical_map)
) #For foreign key

In [ ]:
id_to_canonical.index[df]

In [ ]:
df_sorted[
    df_sorted[id_col].isin(id_to_canonical.index[id_to_canonical=="20250102_100"])
]

In [ ]:
# Get set of canonical IDs
canonical_ids = canonical_map.values
canonical_ids

In [ ]:
# Update primary table: keep only rows with canonical IDs
df_primary = (
    df_primary[df_primary[id_col].isin(canonical_ids)]
    .reset_index(drop=True)
    .drop_duplicates(subset=use_identity_cols + [id_col], keep="first")
)
